# WFP Crop Price Data Collection

This notebook fetches crop price data from the World Food Program's API, processes it, and exports it to Stata format.


In [4]:
import pandas as pd
from pathlib import Path

# Base path. Change this to your dropbox folder
mypc_path = "/Users/justinblack/Library/CloudStorage/Dropbox"

# Data and output folder. These should stay the same
data_folder = f'{mypc_path}/Lavoro_Replication/0_data_construction'  
output_folder = f'{data_folder}/0_data_input/data_processed/crop_prices'

# Create directory from output folder and name the file
output_file_name = "wfp_crop_prices.csv"
output_path = f"{output_folder}/{output_file_name}"

# For debugging, save locally
# output_path = "/Users/justinblack/Desktop/RA_W3/wfp_crop_prices.csv"

# Data Collection

This section downloads crop price data from the World Food Program's Humanitarian Data Exchange (HDX) platform.


In [5]:
# WFP food prices data URLs by country
urls = {
    'gin': "https://data.humdata.org/dataset/17a0a124-bedd-427c-a710-0d6e87672891/resource/f8e11cd4-630f-44be-afa0-3e7baa5744ca/download/wfp_food_prices_gin.csv",
    'lbr': "https://data.humdata.org/dataset/d21bbefd-fdf0-49e9-aa7c-533c32edabf6/resource/85a224fb-5917-43c8-87f7-d14eca7da1e2/download/wfp_food_prices_lbr.csv", 
    'sle': "https://data.humdata.org/dataset/93388727-fdb4-4eb7-8775-9b5b30ae78fa/resource/9249a9f4-6e27-4d52-8603-df302ef862eb/download/wfp_food_prices_sle.csv"
}

# Dictionary to map country codes to full names
country_names = {
    'gin': 'Guinea',
    'lbr': 'Liberia',
    'sle': 'Sierra Leone'
}

# Load data for each country into dataframes and add country columns
dfs = {}
for country, url in urls.items():
    df = pd.read_csv(url)
    df['iso'] = country.upper()
    df['country'] = country_names[country]
    dfs[country] = df

# Combine all dataframes
combined_df = pd.concat(dfs.values(), ignore_index=True)

# Export to .csv
Prepare the data for Stata compatibility and export to .csv format:
1. Convert numeric columns to proper float format
2. Handle string columns appropriately

In [6]:
# Convert relevant numeric columns to float for Stata compatibility
numeric_columns = combined_df.select_dtypes(include=['float64', 'int64']).columns
for col in numeric_columns:
    combined_df[col] = pd.to_numeric(combined_df[col], errors='coerce')

# Convert string columns to proper format for Stata
for col in combined_df.select_dtypes(include='object').columns:
    unique_types = set(type(x) for x in combined_df[col].dropna())
    if all(t in [str, type(None)] for t in unique_types):
        combined_df[col] = combined_df[col].astype(str)
    elif combined_df[col].isnull().all():
        combined_df.drop(columns=[col], inplace=True)
    else:
        print(f"⚠️ Column '{col}' has unsupported types: {unique_types}")
        combined_df.drop(columns=[col], inplace=True)

# Export to Stata format
combined_df.to_csv(output_path, index=False)

print(f"\nData exported at: {output_path}")


Data exported at: /Users/justinblack/Library/CloudStorage/Dropbox/Lavoro_Replication/0_data_construction/0_data_input/data_processed/crop_prices/wfp_crop_prices.csv
